# Notebook 03: DeepSAM: the COVID experiment, solutions

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yangycpku/Machine_Learning_Macro_PSU/blob/main/Tutorials/Tutorial_DeepSAM/notebooks/03_DeepSAM_COVID_Solutions.ipynb)

**Course:** Penn State Mini-Course on Deep Learning and Heterogeneous Agent Macroeconomics (Penn State University, September 15, 2026)
**Session:** Lecture 2 tutorial: Deep Learning for Continuous Time Models and Structural Estimation (DeepSAM)
**Slides:** [`Lectures/Lecture2_slides_Continuous_Time_Structural_Estimation.pdf`](https://github.com/yangycpku/Machine_Learning_Macro_PSU/blob/main/Lectures/Lecture2_slides_Continuous_Time_Structural_Estimation.pdf)
**Notebook role:** solutions to notebook 02
**Runtime:** ~2 min at `smoke` on an A100, ~6 min at `production`
**Author:** Yucheng Yang (University of Zurich and Swiss Finance Institute). [Course repository](https://github.com/yangycpku/Machine_Learning_Macro_PSU)

---

This is notebook 02 with every `TODO` filled in. The economics is the same: the law of
motion of the match distribution, coded by hand and checked against the library, and then
the three results of Section 3 (the COVID calibration, the recovery with and without
re-sorting, and the mechanism behind the gap). Read the filled-in cells against the
equations of notebook 01, Section II; the checks after each exercise confirm that the
hand-written code reproduces the library to floating-point precision.

In [ ]:
RUN_MODE = "smoke"     # one of: "smoke", "teaching", "production"

## Set up the code directory

`src/train_nn.py` holds the whole method: the deterministic steady states, the neural
network, the master-equation residual, the simulation of the distribution, and the training
loop. It expects to be imported with the project root (`Tutorials/Tutorial_DeepSAM`) as the
working directory, because `solve_steady_state` writes its output there as `.npy` files.

* **Google Colab** (the default for this course). The first run clones the course repository
  into `/content` and moves into `Tutorials/Tutorial_DeepSAM`. Colab already ships PyTorch,
  NumPy, SciPy, matplotlib and OmegaConf, so nothing needs to be installed. Use a **GPU
  runtime** (`Runtime -> Change runtime type`): the timings below are for an A100.
* **A local clone.** Open the notebook from inside `Tutorials/Tutorial_DeepSAM/notebooks` and
  the cell steps up to the project root.

The cell also checks that PyTorch can actually launch a kernel on the runtime's GPU. If the
check fails, the GPU is hidden and everything runs on the CPU (slowly). Set
`FORCE_CPU = True` to do that unconditionally. Run this cell **before** any cell that
imports `torch`.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/yangycpku/Machine_Learning_Macro_PSU.git"
REPO_DIR = "/content/Machine_Learning_Macro_PSU"
PROJ_DIR = os.path.join(REPO_DIR, "Tutorials", "Tutorial_DeepSAM")
FORCE_CPU = False   # set True to ignore any GPU and run PyTorch on the CPU

try:
    import google.colab  # noqa: F401  (importable only on a Colab runtime)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if not os.path.isfile(os.path.join(PROJ_DIR, "src", "train_nn.py")):
        print("Cloning the course repository into", REPO_DIR, "...")
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(PROJ_DIR)
elif os.path.isfile("../src/train_nn.py"):
    os.chdir("..")           # a local clone: the notebook lives in notebooks/

if not (os.path.isfile("src/train_nn.py") and os.path.isfile("config/config.yaml")):
    raise FileNotFoundError(
        f"Expected the DeepSAM project root (Tutorials/Tutorial_DeepSAM), but the working "
        f"directory is {os.getcwd()!r}. Open this notebook from inside notebooks/, or "
        f"os.chdir() to the project root."
    )

ROOT = Path.cwd()
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

# GPU check, in a separate process so a failure cannot poison this kernel. If PyTorch
# cannot launch a kernel on the GPU, the GPU is hidden and everything runs on the CPU.
_PROBE = ("import torch; x = torch.randn(8, 8, device='cuda'); "
          "print(torch.cuda.get_device_name(0)); (x @ x).sum().item()")
if FORCE_CPU:
    os.environ["CUDA_VISIBLE_DEVICES"] = ""
    print("FORCE_CPU = True: PyTorch will run on the CPU.")
elif "torch" in sys.modules:
    print("torch is already imported, so the GPU check was skipped "
          "(restart the runtime to run it again).")
else:
    _r = subprocess.run([sys.executable, "-c", _PROBE], capture_output=True, text=True)
    if _r.returncode == 0:
        print("GPU check passed:", _r.stdout.strip().split("\n")[0])
    else:
        _last = [l for l in _r.stderr.strip().split("\n") if l.strip()]
        _last = _last[-1][:160] if _last else "(no error text)"
        if "no CUDA" in _last or "Torch not compiled" in _last or "cuda" in _last.lower() and "available" in _last.lower():
            print("No GPU in this runtime: PyTorch will run on the CPU (slow; use a GPU runtime).")
        else:
            os.environ["CUDA_VISIBLE_DEVICES"] = ""
            print("GPU check FAILED, so the GPU is hidden and PyTorch will run on the CPU.\n  Error was:", _last)

try:
    import omegaconf  # noqa: F401  (preinstalled on Colab; installed here if missing)
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "omegaconf"], check=True)

print("Running on Colab:", IN_COLAB)
print("Project root:", ROOT)

In [ ]:
import contextlib
import inspect
import io
import random
import time
import warnings

import numpy as np
import torch
import torch.optim as optim
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from omegaconf import OmegaConf

from train_nn import Train_NN, Master_PINN_S
import calibration_plot as calplot
import covid_shock_plot as covplot
import plotting

# Two harmless notices from the library and from PyTorch's autograd, silenced for readability.
warnings.filterwarnings("ignore", message=".*To copy construct from a tensor.*")
warnings.filterwarnings("ignore", message=".*no current CUDA context.*")
warnings.filterwarnings("ignore", message=".*pin_memory.*")

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.set_float32_matmul_precision("high")
else:
    print("CPU (no GPU visible) -- everything below still runs, but slowly")

## Choose the run mode

The costs in this notebook are all *simulation* costs: how many paths of the economy are
simulated, and for how long.

| | `smoke` | `teaching` | `production` |
|---|---|---|---|
| pre-COVID ergodic paths × years | 50 × 10 | 100 × 20 | 200 × 30 |
| recovery paths averaged (Figure 2a) | 20 | 60 | 200 |
| wall clock on an A100 | ~2 min | ~3 min | ~6 min |

`production` matches the replication package.

In [ ]:
if RUN_MODE == "smoke":
    ERG_PATHS, ERG_T_END = 50, 10.0
    RECOVERY_PATHS = 20
elif RUN_MODE == "teaching":
    ERG_PATHS, ERG_T_END = 100, 20.0
    RECOVERY_PATHS = 60
elif RUN_MODE == "production":
    ERG_PATHS, ERG_T_END = 200, 30.0
    RECOVERY_PATHS = 200
else:
    raise ValueError(f"Unknown RUN_MODE={RUN_MODE!r}")

print(f"RUN_MODE={RUN_MODE}: {ERG_PATHS} pre-COVID paths over {ERG_T_END:.0f} years, "
      f"{RECOVERY_PATHS} recovery paths")

## Calibration and the deterministic steady states

The calibration and training settings all live in `config/config.yaml`. We load it with
OmegaConf and pass it straight to `Train_NN`, so the object is simply built from that
dictionary.

`solve_steady_state()` then solves the model's **deterministic** steady state once for each
aggregate state $z \in \{L, H, D\}$: low separation (good times), high separation (bad
times), and the disaster state that stands in for COVID. These are fixed points of the
matching problem with the aggregate state frozen. They anchor the aggregate-risk solution,
and the unemployment rates they imply are the first thing to sanity-check against the
calibration.

In [ ]:
cfg = OmegaConf.load(ROOT / "config" / "config.yaml")
params = {
    k: v for k, v in OmegaConf.to_container(cfg.train_nn, resolve=True).items()
    if k != "_target_"
}

seed = int(cfg.seed)
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

ct = Train_NN(**params)
print(f"device {ct.device} | {ct.nx} worker types x {ct.ny} firm types")

t0 = time.monotonic()
with contextlib.redirect_stdout(io.StringIO()):     # the solver is chatty; keep the summary
    ct.solve_steady_state()
print(f"solve_steady_state: {time.monotonic() - t0:.1f}s")

gm_ss = np.load("gm_ss.npy")
gm_low = np.load("gm_low_delta.npy")
gm_high = np.load("gm_high_delta.npy")
gm_dis = np.load("gm_dis_delta.npy")

# State convention (see env.py): z = 0 is L, the good state (separation delta_0 - d_delta),
# z = 1 is H, the bad state (delta_0 + d_delta), z = 2 is D, the disaster state.
for label, gm in [("baseline", gm_ss), ("low separation (L)", gm_low),
                  ("high separation (H)", gm_high), ("disaster (D)", gm_dis)]:
    u = (ct.gw.mean() - np.mean(gm)).cpu().numpy() * 100
    print(f"  unemployment rate, {label:<22s}: {u:6.3f}%")

## Load the trained surplus network

The checkpoint below is the converged network behind the paper's Section 3. Notebook 01
builds this network from scratch and explains what training it involves; here we only need
the solved model.

In [ ]:
pinn_S = Master_PINN_S(
    nn_width=ct.nn_width,
    nn_num_layers=ct.nn_num_layers,
    n_x=ct.nx,
    n_y=ct.ny,
).to(ct.device).float()

ckpt = torch.load(ROOT / "checkpoints" / "section3_surplus_best.pt", map_location=ct.device)
pinn_S.load_state_dict(ckpt["model_state_dict"])
pinn_S.eval()

n_par = sum(p.numel() for p in pinn_S.parameters())
print(f"Loaded the trained surplus network: {ct.nn_num_layers} hidden layers of width "
      f"{ct.nn_width}, {n_par:,} parameters")
print(f"Input dimension: 1 (x) + 1 (y) + 1 (z) + {ct.nx * ct.ny} (g) = {3 + ct.nx * ct.ny}")

## Exercise 1: the law of motion of the match distribution

Everything in the COVID experiment is a simulation of $g_t$ under the trained surplus. From
Section II of notebook 01, between switches of the aggregate state the distribution drifts,

$$
\mu^g(x, y, z, g)
= -\big(\delta(x, y, z) + \varsigma(z)\big)\, g(x, y)
+ \alpha(x, y, z, g)\, m(U, V)\, \frac{g^u(x)}{U}\, \frac{g^v(y)}{V},
$$

and at a switch from $z$ to $\check z$ a fraction $\sigma(z, \check z)$ of matches is destroyed,
$g \mapsto (1 - \sigma(z, \check z))\, g$. Two conventions in the code: $g$ is stored as a
row vector of length $n_x n_y$ in row-major $(x, y)$ order, so `g.reshape(N, ct.nx, ct.ny)`
recovers the grid; and the integrals over types are means over the grids,
$\int g(x, y)\, dy \to \frac{1}{n_y} \sum_j g(x, y_j)$.

The surplus on the whole type grid at the current $(z, g)$ is given (`surplus_on_grid`), and
so is the free-entry block that pins down vacancies $V$ and the vacancy marginal $g^v(y)$
(`ct.marginals_U_V`). Fill in the rest of the drift.

In [ ]:
def surplus_on_grid(g, z_batch):
    """S_hat(x_i, y_j, z, g) on the whole type grid, for each row of g: shape (N, n_x, n_y)."""
    S_grid, _ = ct.calculate_alphas(pinn_S, z_batch, g)
    return S_grid


@torch.no_grad()
def kfe_drift(g, z_idx):
    """KFE drift mu^g(x, y, z, g) for a batch of distributions g: shape (N, n_x n_y), like g.
    z_idx is one aggregate state index (0 = L, 1 = H, 2 = D) shared by the batch."""
    N = g.shape[0]
    z_batch = torch.full((N, 1), float(z_idx), device=ct.device)
    S_grid = surplus_on_grid(g, z_batch)                              # (N, n_x, n_y)

    # TODO (1): acceptance probabilities alpha(x, y, z, g) = 1 / (1 + exp(-xi * S)), shape (N, n_x, n_y).
    #           The curvature parameter xi is ct.xi.
    alphas = 1.0 / (1.0 + torch.exp(-ct.xi * S_grid))

    # TODO (2): worker marginals and aggregate unemployment, from g and the worker mass g_w = ct.gw:
    #           g_e(x) = (1/n_y) sum_y g(x, y);   g_u(x) = g_w(x) - g_e(x);   U = (1/n_x) sum_x g_u(x).
    #           Shapes: g_e, g_u (N, n_x); U (N, 1).
    g_e = g.reshape(N, ct.nx, ct.ny).mean(dim=2)                  # (N, n_x)
    g_u = ct.gw - g_e                                            # (N, n_x)
    U = g_u.mean(dim=1, keepdim=True)                            # (N, 1)

    # Free entry pins down vacancies: V and the vacancy marginal g_v(y) come from the library.
    _, _, _, g_p, g_v, V, g_f = ct.marginals_U_V(g, S_grid, alphas, z_batch)   # g_v: (N, n_y), V: (N, 1)

    # TODO (3): meeting rate per unemployed worker, M_u = m(U, V) / U, with the Cobb-Douglas
    #           meeting function m(U, V) = kappa U^nu V^(1 - nu)  (ct.kappa, ct.nu). Shape (N, 1).
    M_u = ct.kappa * U ** ct.nu * V ** (1 - ct.nu) / U

    # TODO (4): the drift, shape (N, n_x n_y):
    #           mu(x, y) = -(delta(x, y, z) + varsigma(z)) g(x, y) + (M_u / V) alpha(x, y) g_u(x) g_v(y).
    #           ct.delta_s[z_idx] is delta(x, y, z) on the grid, shape (n_x, n_y); ct.varsigma_s[z_idx] is varsigma(z).
    #           Hint: the outer product g_u(x) g_v(y) is g_u[:, :, None] * g_v[:, None, :].
    delta = ct.delta_s[z_idx].reshape(1, -1)                     # (1, n_x n_y)
    varsigma = ct.varsigma_s[z_idx]
    inflow = (M_u / V) * alphas.reshape(N, -1) * (g_u[:, :, None] * g_v[:, None, :]).reshape(N, -1)
    mu = -(delta + varsigma) * g + inflow
    return mu

In [ ]:
# Check against the library's drift (ct.mu_g), at the baseline steady state in each aggregate state.
g_test = torch.tensor(gm_ss, dtype=torch.float32, device=ct.device).reshape(1, -1)
for z_idx, name in [(0, "L"), (1, "H"), (2, "D")]:
    z_batch = torch.full((1, 1), float(z_idx), device=ct.device)
    with torch.no_grad():
        S_grid, alphas = ct.calculate_alphas(pinn_S, z_batch, g_test)
        g_e, g_u, U, g_p, g_v, V, g_f = ct.marginals_U_V(g_test, S_grid, alphas, z_batch)
        mu_lib = ct.mu_g(g_test, ct.m(U, V) / U, V, alphas, g_e, g_p, g_f, z_batch)
        mu_nb = kfe_drift(g_test, z_idx)
    assert torch.allclose(mu_nb, mu_lib, atol=1e-6, rtol=1e-5), f"drift differs from ct.mu_g in state {name}"
    print(f"state {name}: max |mu_notebook - mu_library| = {(mu_nb - mu_lib).abs().max().item():.2e}   OK")
print("The baseline steady state is a fixed point of the good-state dynamics only approximately: "
      f"mean |mu| in L = {kfe_drift(g_test, 0).abs().mean().item():.2e}")

Now the time loop. Between two grid times the drift is integrated with an Euler step
(`substeps` micro-steps per $dt$), the distribution is clamped at zero, and at a switch of
the aggregate state the jump is applied *exactly at the switch*, before recording. If the
economy enters the path from a different state than the path's first entry (`z_pre`), the
same jump is applied at $t = 0$, which is why the jump comes first in the numbering. The function records aggregate employment
$P_t = \frac{1}{n_x n_y}\sum_{x,y} g_t(x, y)$ at every grid time.

In [ ]:
@torch.no_grad()
def apply_jump(g, z_from, z_to):
    """The post-switch distribution when the aggregate state jumps from z_from to z_to."""
    # TODO (5): g <- (1 - sigma(z_from, z_to)) g, with the jump-exit matrix ct.sigma_mat (3 x 3, indexed [z_from, z_to]).
    return (1.0 - ct.sigma_mat[z_from, z_to]) * g


@torch.no_grad()
def simulate_g(g0, z_path, dt=0.01, substeps=1, z_pre=None):
    """Simulate g_t along one path of the aggregate state.
    g0: (1, n_x n_y); z_path: 1-D sequence of state indices at t = 0, dt, 2 dt, ...;
    z_pre: the state the economy is in just before t = 0 (a jump is applied at t = 0 if it differs).
    Returns the final g and the path of aggregate employment P_t (one entry per grid time)."""
    z_path = [int(round(float(z))) for z in z_path]
    g = g0.clone()
    if z_pre is not None and int(round(float(z_pre))) != z_path[0]:
        g = apply_jump(g, int(round(float(z_pre))), z_path[0]).clamp(min=0.0)
    P = [g.mean().item()]
    for k in range(len(z_path) - 1):
        for _ in range(substeps):
            # TODO (6): one Euler micro-step of the KFE drift under the current state z_path[k], then clamp at zero:
            #           g <- max(g + (dt / substeps) * mu^g(g, z_path[k]), 0)
            g = torch.clamp(g + (dt / substeps) * kfe_drift(g, z_path[k]), min=0.0)
        if z_path[k + 1] != z_path[k]:
            g = apply_jump(g, z_path[k], z_path[k + 1]).clamp(min=0.0)
        P.append(g.mean().item())
    return g, np.array(P)

In [ ]:
# Check against the library's single-path simulator on a path that exercises every branch:
# start in H, jump into D at t = 0, stay for 0.2 years, then jump back to H and recover.
z_check = [2] * 20 + [1] * 60
g0 = torch.tensor(gm_high, dtype=torch.float32, device=ct.device).reshape(1, -1)
g_end, P_nb = simulate_g(g0, z_check, dt=0.01, substeps=1, z_pre=1)
with contextlib.redirect_stdout(io.StringIO()):
    out_lib = covplot.simulate_path(
        ct=ct, pinn_S=pinn_S,
        z_path=torch.tensor([z_check], dtype=torch.float32, device=ct.device),
        g_init=g0.clone(), substeps=1, z_pre=1.0, clamp_g=True, k_mu=2, dt=0.01,
    )
P_lib = np.asarray(out_lib["P"], dtype=float)
assert P_nb.shape == P_lib.shape and np.allclose(P_nb, P_lib, atol=1e-6, rtol=1e-5), "simulate_g differs from covplot.simulate_path"
print(f"max |P_notebook - P_library| over {len(P_nb)} grid times = {np.abs(P_nb - P_lib).max():.2e}   OK")
print(f"employment falls from {100 * P_nb[0]:.2f}% at the jump into D to {100 * P_nb[19]:.2f}% after 0.2 years, "
      f"then recovers to {100 * P_nb[-1]:.2f}% after 0.6 years in H")

## Exercise 2: Figure 1, calibrating the disaster state

Before the shock, the economy sits in the ergodic distribution of the *two-state* $L/H$
chain; the disaster state is off. `ergodic_g_LH_ctmc` (given) simulates that chain forward
and averages the match distribution after a burn-in: this is $\bar g_0$, the pre-COVID
distribution, and $z_{\text{pre}}$ is the state the economy is in when the shock hits.

The shock is a jump into the disaster state $D$ followed by 0.2 years of drift there. The
disaster separation rate $\delta(x, y, D)$ was calibrated so that the resulting employment
decline by worker type and by firm type matches spring 2020. Compute those declines from the
distribution before and after the shock.

In [ ]:
t0 = time.monotonic()
with contextlib.redirect_stdout(io.StringIO()):
    g0_bar, stats0, z_paths_noD = calplot.ergodic_g_LH_ctmc(
        ct=ct, pinn_S=pinn_S, g_init=None, N_paths=ERG_PATHS, dt=0.01,
        T_end=ERG_T_END, burn_in=ERG_T_END / 3, record_interval=10,
        substeps=1, clamp_g=True, seed=123,
    )
z_pre_paths = z_paths_noD[:, -1]                       # aggregate state just before COVID, per path
z_pre_scalar = float(covplot.to_numpy(z_paths_noD)[0, -1])
print(f"pre-COVID ergodic distribution over {ERG_PATHS} paths: {time.monotonic() - t0:.1f}s; "
      f"the shock hits in state {'L' if z_pre_scalar == 0 else 'H'}; "
      f"pre-COVID unemployment {100 * stats0['U']:.2f}%")

In [ ]:
def emp_worker(g):
    """Employed mass by worker type, e^w(x) = (1/n_y) sum_y g(x, y): shape (n_x,)."""
    return g.reshape(ct.nx, ct.ny).mean(dim=1)

def emp_firm(g):
    """Employed mass by firm type, e^f(y) = (1/n_x) sum_x g(x, y): shape (n_y,)."""
    return g.reshape(ct.nx, ct.ny).mean(dim=0)


T_SHOCK = 0.2                                              # years spent in the disaster state
g_pre = torch.as_tensor(g0_bar, dtype=torch.float32, device=ct.device).reshape(1, -1)
z_shock = [2] * (int(round(T_SHOCK / 0.01)) + 1)           # D at t = 0, dt, ..., 0.2
g_post, P_shock = simulate_g(g_pre, z_shock, dt=0.01, substeps=1, z_pre=z_pre_scalar)

# TODO (7): the percentage decline in employment by worker type and by firm type over the shock,
#           (e_pre - e_post) / e_pre, using emp_worker and emp_firm above. Shapes (n_x,) and (n_y,).
worker_drop = 1.0 - emp_worker(g_post) / emp_worker(g_pre)
firm_drop = 1.0 - emp_firm(g_post) / emp_firm(g_pre)

target_worker = np.array([0.372, 0.236, 0.180, 0.141, 0.087])
target_firm = np.array([0.138, 0.012, 0.121, 0.165, 0.126, 0.177, 0.175, 0.198, 0.21, 0.289, 0.325])[::-1]
model_worker = worker_drop.cpu().numpy()
model_firm = firm_drop.cpu().numpy()

fig, axes = plt.subplots(1, 2, figsize=(11, 4), dpi=120)
for ax, model, target, name in [(axes[0], model_worker, target_worker, "worker type $x$"),
                                (axes[1], model_firm, target_firm, "firm type $y$")]:
    xb = np.arange(len(model)) / (len(model) - 1)
    ax.bar(xb, model * 100, width=0.04, color="tab:blue", label="model")
    ax.bar(xb, target * 100, width=0.02, color="tab:red", label="calibration target")
    ax.set_xlabel(name); ax.set_ylabel("employment decline (%)"); ax.legend(fontsize=9)
axes[0].set_title("Employment decline by worker type"); axes[1].set_title("Employment decline by firm type")
plt.tight_layout(); plt.show()
print(f"model employment decline: {model_worker[0]:.1%} for the lowest worker type, "
      f"{model_worker[-1]:.1%} for the highest (targets: {target_worker[0]:.1%} and {target_worker[-1]:.1%})")

In [ ]:
# Check against the library's Figure-1 simulator, run on the same single pre-COVID path.
with contextlib.redirect_stdout(io.StringIO()):
    fig1_out = calplot.simulate_disaster_0p2_for_figure1(
        ct=ct, pinn_S=pinn_S, g0_paths=g_pre, z_pre_paths=torch.tensor([int(z_pre_scalar)]),
        dt=0.01, T_shock=T_SHOCK, substeps=1, clamp_g=True,
    )
assert np.allclose(model_worker, fig1_out["worker_emp_drop_pct_mean"].numpy(), atol=1e-5), "worker declines differ from the library"
assert np.allclose(model_firm, fig1_out["firm_emp_drop_pct_mean"].numpy(), atol=1e-5), "firm declines differ from the library"
print("employment declines by type == library: OK")

## Exercise 3: Figure 2a, what the distribution does

Two economies, the same shock and the same aggregate path: 0.2 years in the disaster state,
then the bad state $H$ (held fixed here, so that the only randomness is in the shock's
timing). One is free to re-sort: after the disaster, new matches form wherever the surplus is
positive given the *current* distribution. That is your `simulate_g`. The other is held to
its pre-COVID composition (`simulate_figure3a_orange_one_path`, given). The difference
between the two paths is the distributional feedback.

Employment is reported relative to the deterministic steady state of the bad state, so that
100 is "back to normal for state $H$".

In [ ]:
DT, T_END, T_COVID = 0.01, 2.0, 0.2
n_covid = int(round(T_COVID / DT))
n_total = int(round(T_END / DT))
z_post = covplot.generate_single_LH_path_from_start(ct=ct, start_state="H", T_steps=n_total - n_covid,
                                                    dt=DT, seed=123, fixed_z=True)
z_full = np.concatenate([np.full(n_covid, 2.0), covplot.to_numpy(z_post)[0]])     # D, then H
t_grid = DT * np.arange(n_total)

g_end, P_path = simulate_g(g_pre, z_full, dt=DT, substeps=1, z_pre=z_pre_scalar)

# TODO (8): employment relative to the bad-state steady state, in percent: 100 * P_t / P_ref,
#           where P_ref is aggregate employment in the H steady state, the mean of gm_high.
P_ref = float(np.mean(gm_high))
rel_emp = 100.0 * P_path / P_ref

res_blue = {"t": t_grid, "rel_emp": rel_emp, "z_path": z_full}
with contextlib.redirect_stdout(io.StringIO()):
    res_orange = covplot.simulate_figure3a_orange_one_path(
        ct=ct, pinn_S=pinn_S, gm_low=gm_low, gm_high=gm_high, g0_bar=g0_bar,
        z_pre_scalar=z_pre_scalar, z_path_full=z_full, substeps=1, clamp_g=True,
    )
plotting.plot_relative_employment_2a(res_blue, res_orange)
print(f"after {T_END:.0f} years: employment {rel_emp[-1]:.2f}% of normal with re-sorting, "
      f"{res_orange['rel_emp'][-1]:.2f}% when held to the pre-COVID composition")

In [ ]:
# Check against the library's full-dynamics path.
with contextlib.redirect_stdout(io.StringIO()):
    res_lib = covplot.simulate_figure3a_blue_one_path(
        ct=ct, pinn_S=pinn_S, gm_low=gm_low, gm_high=gm_high, g0_bar=g0_bar,
        z_pre_scalar=z_pre_scalar, z_after_start="H", dt=DT, T_end=T_END, t_covid=T_COVID,
        substeps=1, clamp_g=True, seed=123, k_mu=3, fixed_z=True,
    )
assert np.allclose(rel_emp, res_lib["rel_emp"], atol=1e-4), "relative employment differs from the library"
print("full-dynamics recovery path == library: OK")

### Averaging over recovery paths

One path is a story; the average over many is the result. Here the post-COVID aggregate
state is random again, and both economies are averaged over `RECOVERY_PATHS` independent
recovery paths. This is the most expensive cell in the notebook.

In [ ]:
t0 = time.monotonic()
with contextlib.redirect_stdout(io.StringIO()):
    final_out = covplot.plot_figure3a_average_many_paths(
        ct=ct, pinn_S=pinn_S, g0_bar=g0_bar, z_pre_scalar=z_pre_scalar,
        z_after_start="H", N_recovery_paths=RECOVERY_PATHS, dt=0.01, T_end=2.0,
        t_covid=0.2, substeps=1, clamp_g=True, seed_recovery_paths=123,
    )
print(f"{RECOVERY_PATHS} recovery paths: {time.monotonic() - t0:.1f}s")
plotting.plot_relative_employment_many(final_out)

## Figure 3: where the feedback comes from

The acceptance sets and the composition of the match distribution, before and after the
shock. This is the mechanism behind the gap in the previous figure: the disaster destroys
matches selectively, which changes *which* new matches are worth forming, which changes the
speed of the recovery.

In [ ]:
fig4_sep = plotting.plot_figure4_panels_separately(
    ct=ct, pinn_S=pinn_S, z_ergodic_for_alpha="L",
    dpi=120, panel_ratio=800 / 543, fig_height=4.0, panels=("a", "b", "c", "d"),
)

## Summary

* The law of motion of $g_t$ is a few lines once $\widehat{S}$ is known: acceptance from the
  surplus, marginals from the distribution, meetings from the matching function, the KFE
  drift, and a jump at every switch of the aggregate state. You wrote them and they
  reproduce the library.
* The disaster state is calibrated to the observed spring-2020 employment decline by worker
  and firm type, a two-sided target that a model without two-sided heterogeneity could not
  hit.
* Letting the economy re-sort after the shock produces a materially different recovery from
  holding the match distribution fixed. That gap is the distributional feedback, and it is
  only computable because the solution carries $g$ in the state.

## Takeaway

The distribution is not a bookkeeping device here: it changes the aggregate path. The
methodological cost of that is solving a master equation in 55 state dimensions, which is
what the neural network of notebook 01 buys you.